This a notebook for the Digit Recognizer Kaggle competition. This notebook has a leaderboard score of 0.99203. 

We start by exploring the data: from the distribution of numbers to what they look like.

We then generate more data by zooming, rotating, and shifting the data to improve the CNNs robustness to variation. 

We construct a CNN using three layers of feature learning followed by flattening layer that is then fed to the classifier head. We use the standard 'relu' for activation and 'softmax' for the head. 

We compile the model using adam as the optimizer and check performance via sparse categorical crossentropy for loss as well as accuracy. We employ an early stopping protocol based on validation loss in order to save time and memory. 


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

Let's load the data andd get an idea of what the raw CSV file looks like

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load data
train = pd.read_csv('/kaggle/input/competitions/digit-recognizer/train.csv')
test = pd.read_csv('/kaggle/input/competitions/digit-recognizer/test.csv')

print(train.shape)
print(test.shape)
print(train.head())

Looks like the first column is the number in the image and the other 784 columns contain pixel information. 

Now we seperate the labels and features, normalize the pixels, and reshape the data so as to make it easy to convert to a 2D image. 

In [ ]:
# Separate labels and features
y_train = train['label'].values
X_train = train.drop('label', axis=1).values
X_test = test.values

# Normalize pixel values from 0-255 to 0-1
X_train = X_train / 255.0
X_test = X_test / 255.0

# Reshape to 28x28 images
X_train = X_train.reshape(-1, 28, 28)
X_test = X_test.reshape(-1, 28, 28)

print(f"Training shape: {X_train.shape}")
print(f"Labels shape: {y_train.shape}")
print(f"Test shape: {X_test.shape}")

And now let's get an idea of what we are looking at

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))

for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i], cmap='gray')
    ax.set_title(f'Label: {y_train[i]}')
    ax.axis('off')

plt.tight_layout()
plt.show()

As expected: a variety of handwritten numbers. Now let's check the distribution of labels, just to make sure we don't have to weight the data

In [ ]:
import collections

counts = collections.Counter(y_train)
plt.bar(counts.keys(), counts.values())
plt.xlabel('Digit')
plt.ylabel('Count')
plt.title('Class Distribution')
plt.xticks(range(10))
plt.show()

print(counts)

For the most part, the labels are fairly distributed. Perhaps the difference between the frequency of '1' and '5' is cause for concern, but it's probably trouble than it's worth. 

Now let's add the grayscale to the features

In [ ]:
# Add channel dimension for CNN
X_train = X_train.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)

print(f"Training shape: {X_train.shape}")  # (42000, 28, 28, 1)

Now we will want to preprocess the images a bit in order to solidify the robustness of the CNN. We will do vertical and horizontal shifts, zooming, and small rotations so as to not confuse '6' and '9' and also to keep the adjusted data reasonably recognizable as a number. 

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1
)

datagen.fit(X_train)

Now we will split the training data in to training and validation sets, with a 80/20 split. 

In [ ]:
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)


Finally, we construct the CNN. We have three layers of feature extraction which use a convolutional layer and a maxpooling layer. We use 'relu' as our activation function for the convolution and dense layers. We employ a dropout layer to reduce the chance of overfitting and finish with a dense softmax layer using softmax to get probabilities for each number. 

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(128, (3,3), activation='relu'),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

model.summary()

Now it's time to train the CNN on the data. To compile, we will use ADAM as the optimizer and calculate loss using sparse_categorical_crossentropy. We also monitor performance using accuracy. We employ early stopping based on validation loss in order to save time and memory. 

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    datagen.flow(X_tr, y_tr, batch_size=64),
    epochs=50,
    validation_data=(X_val, y_val),
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Accuracy
axes[0].plot(history.history['accuracy'], label='Training')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

# Loss
axes[1].plot(history.history['loss'], label='Training')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.show()

Looks like we hardly need about half the epochs to get a good result. Both the training and validation curves flatten out quickly and there is no sign of over or under fitting

Now we make our predictions and submit our results to the competition. 

In [ ]:
# Generate predictions
predictions = model.predict(X_test)
predictions = np.argmax(predictions, axis=1)

# Create submission file
submission = pd.DataFrame({
    'ImageId': range(1, len(predictions) + 1),
    'Label': predictions
})

submission.to_csv('submission.csv', index=False)
print(submission.head())